# Full Comparison Experiment on Google Colab

This notebook runs the complete base vs finetuned model comparison and automatically saves results to Google Drive.

**What this does:**
1. Mounts Google Drive for automatic result saving
2. Clones the repository (if needed)
3. Installs dependencies
4. Downloads datasets
5. Runs the full comparison experiment
6. Saves all results to Drive automatically

**Runtime:** ~2-4 hours with GPU (T4)

**Before starting:**
- Go to Runtime → Change runtime type → Select GPU

## Step 1: Mount Google Drive

In [ ]:
from google.colab import drive
import os

# Mount Drive (you'll need to click "Allow" once)
drive.mount('/content/drive')

# Create results directory in Drive
DRIVE_RESULTS_DIR = '/content/drive/MyDrive/Negation-Origin-Tracing-Results'
os.makedirs(DRIVE_RESULTS_DIR, exist_ok=True)

print(f"✓ Drive mounted successfully")
print(f"✓ Results will be saved to: {DRIVE_RESULTS_DIR}")

## Step 2: Clone Repository

In [ ]:
import os

REPO_DIR = '/content/Negation-Origin-Tracing'

if not os.path.exists(REPO_DIR):
    print("Cloning repository...")
    !git clone https://github.com/TheMattWang/Negation-Origin-Tracing.git
    print("✓ Repository cloned")
else:
    print("✓ Repository already exists")
    # Pull latest changes
    !cd {REPO_DIR} && git pull

# Change to repo directory
%cd {REPO_DIR}
!pwd

## Step 3: Install Dependencies

In [ ]:
# Install required packages
!pip install -q torch lightning transformers datasets pandas pyarrow matplotlib seaborn scikit-learn tqdm tensorboardX

print("\n✓ Dependencies installed")

# Verify GPU availability
import torch
if torch.cuda.is_available():
    print(f"✓ GPU detected: {torch.cuda.get_device_name(0)}")
    print(f"  Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
else:
    print("⚠ No GPU detected - training will be much slower!")
    print("  Go to Runtime → Change runtime type → Select GPU")

## Step 4: Download Datasets

In [ ]:
import os

# Check if data already exists
if os.path.exists('data/raw/train/train.parquet'):
    print("✓ Datasets already downloaded")
else:
    print("Downloading datasets...")
    !python src/data/download.py
    print("✓ Datasets downloaded")

# Verify data
!ls -lh data/raw/train/
!ls -lh data/raw/test/

## Step 5: Run Full Comparison Experiment

This will:
1. Train probes on base model (all layers)
2. Identify best layer
3. Run interventions on base model
4. Run interventions on finetuned model
5. Generate comparison summary

**This takes 2-4 hours with GPU!**

In [ ]:
import os
from datetime import datetime

# Set environment variable to save to Drive
os.environ['DRIVE_OUTPUT'] = DRIVE_RESULTS_DIR

print(f"Starting experiment at {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print(f"Results will be saved to: {DRIVE_RESULTS_DIR}")
print("\n" + "="*60)

# Run the full comparison script
!bash run_full_comparison.sh

## Step 6: Verify Results Saved to Drive

In [ ]:
import glob
import os
from datetime import datetime

# Find the most recent comparison folder in Drive
comparison_dirs = glob.glob(f'{DRIVE_RESULTS_DIR}/comparison_*')

if comparison_dirs:
    latest = max(comparison_dirs, key=os.path.getmtime)
    folder_name = os.path.basename(latest)
    
    print(f"✓ Experiment completed!")
    print(f"\nResults saved to Google Drive:")
    print(f"  {latest}")
    print(f"\nFolder structure:")
    !ls -lh {latest}
    
    # Check for key files
    print(f"\nKey result files:")
    if os.path.exists(f"{latest}/comparison_summary.json"):
        print(f"  ✓ comparison_summary.json")
    if os.path.exists(f"{latest}/base_probes/results_summary.json"):
        print(f"  ✓ base_probes/results_summary.json")
    if os.path.exists(f"{latest}/base_interventions/intervention_results.json"):
        print(f"  ✓ base_interventions/intervention_results.json")
    if os.path.exists(f"{latest}/finetuned_interventions/intervention_results.json"):
        print(f"  ✓ finetuned_interventions/intervention_results.json")
    
    print(f"\n" + "="*60)
    print(f"To access results:")
    print(f"1. Open Google Drive on your computer")
    print(f"2. Navigate to: My Drive/Negation-Origin-Tracing-Results/{folder_name}")
    print(f"3. Download or sync to your local machine")
    print(f"="*60)
else:
    print("⚠ No comparison folders found in Drive")
    print("Check if the experiment completed successfully above")

## Optional: Quick Results Preview

In [ ]:
import json
import glob
import os

# Find latest results
comparison_dirs = glob.glob(f'{DRIVE_RESULTS_DIR}/comparison_*')
if comparison_dirs:
    latest = max(comparison_dirs, key=os.path.getmtime)
    summary_file = f"{latest}/comparison_summary.json"
    
    if os.path.exists(summary_file):
        with open(summary_file, 'r') as f:
            summary = json.load(f)
        
        print("Experiment Summary")
        print("="*60)
        print(f"\nBase Model: {summary['experiment_config']['base_model']}")
        print(f"Finetuned Model: {summary['experiment_config']['finetuned_model']}")
        print(f"\nBest Layer: {summary['experiment_config']['best_layer']}")
        print(f"Best Pooling: {summary['experiment_config']['best_pooling']}")
        
        if summary.get('probe_results'):
            print(f"\nProbe Results:")
            print(f"  Total experiments: {summary['probe_results']['total_experiments']}")
            print(f"  Best AUROC: {summary['probe_results']['best_auroc']:.4f}")
        
        print("\n" + "="*60)
        print("Full results available in Google Drive")
    else:
        print("Summary file not found - experiment may still be running")
else:
    print("No results found yet")

## Optional: Download Results as ZIP (Backup)

If you want a local backup in addition to Drive:

In [ ]:
import glob
import os

# Find latest results
comparison_dirs = glob.glob(f'{DRIVE_RESULTS_DIR}/comparison_*')
if comparison_dirs:
    latest = max(comparison_dirs, key=os.path.getmtime)
    folder_name = os.path.basename(latest)
    
    # Create zip file
    zip_name = f"/content/{folder_name}.zip"
    print(f"Creating zip file: {zip_name}")
    !cd {DRIVE_RESULTS_DIR} && zip -r {zip_name} {folder_name}
    
    print(f"\n✓ Zip file created: {zip_name}")
    print(f"\nTo download:")
    print(f"1. Click the folder icon on the left (Files)")
    print(f"2. Find {folder_name}.zip")
    print(f"3. Right-click → Download")
    
    # Show file size
    !ls -lh {zip_name}
else:
    print("No results to zip")